# Context Enrichment Window（Window-around-chunk）：检索到“点”，再补齐“前后上下文”

本节目标：实现原仓库的 *window-around-chunk* 最小版本，并理解它与 RSE 的区别：

- **RSE**：先给整篇文档所有 chunks 打分 → 在序列上找连续高分段（segment）
- **Window-around-chunk**：先检索到一个（或少数几个）最相关 chunk → 直接取它的前后邻居 chunks 拼回去

它特别适合这类场景：

- 检索命中的是“句子中间/表格中间/定义中间”，单块读不完整
- 你希望“以命中的 chunk 为中心”，固定窗口地扩展上下文

## 0) 导入依赖并加载环境变量

延续前面 notebook 的约定：从 `../.env` 读取 `OPENAI_API_KEY`。

这节会用：

- `langchain_openai.OpenAIEmbeddings` 做向量
- `FAISS` 做向量库
- `RecursiveCharacterTextSplitter(add_start_index=True)` 切分并记录原文位置


In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path
from typing import List

import numpy as np
import requests
from dotenv import load_dotenv

load_dotenv("../.env")

# 让 `all_rag_techniques/` 子目录里的 notebook
# 也能导入上一级目录的 `helper_functions.py`
sys.path.insert(0, str(Path("..").resolve()))

from pypdf import PdfReader

from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

from helper_functions import show_context

/tmp/ipykernel_3960568/2154944116.py:23: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


## 1) 下载 PDF 并抽取为纯文本

我们下载原仓库同款文件：`Understanding_Climate_Change.pdf`。然后用 `pypdf` 逐页提取文本并拼起来。

这一节的重点不在“PDF 解析质量”，而是在“检索后补齐邻居上下文”的 retrieval 后处理逻辑。

In [2]:
os.environ["HTTP_PROXY"] = "http://127.0.0.1:7890"
os.environ["HTTPS_PROXY"] = "http://127.0.0.1:7890"
os.environ["ALL_PROXY"] = "http://127.0.0.1:7890"

In [3]:
DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

PDF_URL = "https://raw.githubusercontent.com/NirDiamant/RAG_Techniques/main/data/Understanding_Climate_Change.pdf"
PDF_PATH = DATA_DIR / "Understanding_Climate_Change.pdf"

if not PDF_PATH.exists():
    resp = requests.get(
        PDF_URL,
        timeout=60,
        headers={"User-Agent": "Mozilla/5.0"},
    )
    resp.raise_for_status()
    PDF_PATH.write_bytes(resp.content)

reader = PdfReader(str(PDF_PATH))
pages_text = [(p.extract_text() or "") for p in reader.pages]
content = "\n\n".join(pages_text)

assert len(content.strip()) > 0, "PDF 提取到的文本为空（pypdf.extract_text 可能失败）"
print("Chars:", len(content))
print("Pages:", len(reader.pages))

Chars: 72592
Pages: 33


## 2) 切分为带重叠的 chunks，并给每个 chunk 标记顺序编号

这一节和 RSE 有个关键不同：

- RSE 要求 `chunk_overlap=0`（方便拼“连续段”）
- Window-around-chunk **可以有 overlap**（因为我们只是在一个命中点周围扩窗）

我们会把每个 chunk 保存为 `Document`，并在 `metadata` 里加入：

- `chunk_index`：它在原文中的顺序编号（0,1,2,...）
- `start_index`：chunk 在原文字符串中的起始位置（LangChain splitter 自动加）

后面扩窗时，我们只需要根据 `chunk_index` 去拿邻居 `chunk_index±N` 的文本。

In [4]:
CHUNK_SIZE = 400
CHUNK_OVERLAP = 200

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    add_start_index=True,
)

docs = splitter.create_documents([content])
for i, d in enumerate(docs):
    d.metadata["chunk_index"] = i
    d.metadata["source"] = str(PDF_PATH)

print("Num chunks:", len(docs))
print("Example metadata:", docs[0].metadata)
print("\n--- Chunk 0 preview ---\n")
print(docs[0].page_content[:500])

Num chunks: 366
Example metadata: {'start_index': 0, 'chunk_index': 0, 'source': '../data/Understanding_Climate_Change.pdf'}

--- Chunk 0 preview ---

Understanding Climate Change 
Chapter 1: Introduction to Climate Change 
Climate change refers to significant, long-term changes in the global climate. The term 
"global climate" encompasses the planet's overall weather patterns, including temperature, 
precipitation, and wind patterns, over an extended period. Over the past century, human


## 3) 建立向量库与 retriever

LangChain V1的用法是把 vectorstore 转成 retriever，然后用：

- `retriever.invoke(query)` 获取 `List[Document]`

In [5]:
emb = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(docs, emb)
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

## 4) 核心：检索命中 chunk 后，把前后邻居 chunk 拼成“窗口上下文”

我们实现一个最小函数 `retrieve_with_context_window`：

1. 用 `retriever.invoke(query)` 命中最相关 chunk（k=1）
2. 取出它的 `chunk_index`
3. 取 `[chunk_index-num_neighbors, ..., chunk_index+num_neighbors]` 的邻居 chunks
4. 按顺序拼接成一个更宽的上下文字符串

因为 chunks 有 overlap，所以拼接时会去掉重复的重叠部分（教学版用固定 `CHUNK_OVERLAP` 字符裁剪）。

In [6]:
def _concat_with_fixed_overlap(texts: List[str], *, chunk_overlap: int) -> str:
    if not texts:
        return ""
    out = texts[0]
    for t in texts[1:]:
        cut = max(0, len(out) - chunk_overlap)
        out = out[:cut] + t
    return out


def retrieve_with_context_window(
    docs_by_index: List[Document],
    retriever,
    query: str,
    *,
    num_neighbors: int = 1,
    chunk_overlap: int,
) -> List[str]:
    hits = retriever.invoke(query)
    sequences: List[str] = []

    for doc in hits:
        idx = doc.metadata.get("chunk_index")
        assert isinstance(idx, int), "retrieved doc missing metadata['chunk_index']"

        start = max(0, idx - num_neighbors)
        end = min(len(docs_by_index), idx + num_neighbors + 1)

        window_docs = docs_by_index[start:end]
        window_texts = [d.page_content for d in window_docs]
        sequences.append(_concat_with_fixed_overlap(window_texts, chunk_overlap=chunk_overlap))

    return sequences

## 5) 对比：普通检索 vs 扩窗检索

我们用原仓库同款风格：

- baseline：只看命中的 chunk
- enriched：命中 chunk + 前后各 1 个邻居（`num_neighbors=1`）

你会看到 enriched 的上下文通常更连贯，尤其当 baseline 刚好命中在句子/段落中间时。

In [7]:
query = "Explain the role of deforestation and fossil fuels in climate change."

baseline_docs = retriever.invoke(query)
baseline_texts = [d.page_content for d in baseline_docs]

enriched_texts = retrieve_with_context_window(
    docs,
    retriever,
    query,
    num_neighbors=1,
    chunk_overlap=CHUNK_OVERLAP,
)

print("Regular retrieval (baseline):\n")
show_context(baseline_texts)

print("Retrieval with context window:\n")
show_context(enriched_texts)

Regular retrieval (baseline):

Context 1:
activities, particularly the burning of fossil fuels and deforestation, have significantly 
contributed to climate change. 
Historical Context 
The Earth's climate has changed throughout history. Over the past 650,000 years, there have 
been seven cycles of glacial advance and retreat, with the abrupt end of the last ice age about

Retrieval with context window:

Context 1:
"global climate" encompasses the planet's overall weather patterns, including temperature, 
precipitation, and wind patternactivities, particularly the burning of fossil fuels and deforestation, have significantly 
contributed to climate change. 
HistoricThe Earth's climate has changed throughout history. Over the past 650,000 years, there have 
been seven cycles of glacial advance and retreat, with the abrupt end of the last ice age about 
11,700 years ago marking the beginning of the modern climate era and human civilization. 
Most of these climate changes are attributed t

## 6) 一个更直观的玩具例子（对齐原仓库）

为了更容易看到优势，我们用一个手写的小文档：

- baseline 可能只命中一句话的一半
- window-around-chunk 会把前后邻居补回来，让答案更完整

这个例子的目的：让你在 *输出文本层面* 看懂“补窗”的价值。

In [8]:
toy_text = """
Artificial Intelligence (AI) has a rich history dating back to the mid-20th century. The term "Artificial Intelligence" was coined in 1956 at the Dartmouth Conference, marking the field's official beginning.

In the 1950s and 1960s, AI research focused on symbolic methods and problem-solving. The Logic Theorist, created in 1955 by Allen Newell and Herbert A. Simon, is often considered the first AI program.

The 1960s saw the development of expert systems, which used predefined rules to solve complex problems. DENDRAL, created in 1965, was one of the first expert systems, designed to analyze chemical compounds.

However, the 1970s brought the first "AI Winter," a period of reduced funding and interest in AI research, largely due to overpromised capabilities and underdelivered results.

The 1980s saw a resurgence with the popularization of expert systems in corporations. The Japanese government's Fifth Generation Computer Project also spurred increased investment in AI research globally.

Neural networks gained prominence in the 1980s and 1990s. The backpropagation algorithm, although discovered earlier, became widely used for training multi-layer networks during this time.

The late 1990s and 2000s marked the rise of machine learning approaches. Support Vector Machines (SVMs) and Random Forests became popular for various classification and regression tasks.

Deep Learning, a subset of machine learning using neural networks with many layers, began to show promising results in the early 2010s. The breakthrough came in 2012 when a deep neural network significantly outperformed other machine learning methods in the ImageNet competition.

Since then, deep learning has revolutionized many AI applications, including image and speech recognition, natural language processing, and game playing. In 2016, Google's AlphaGo defeated a world champion Go player, a landmark achievement in AI.
""".strip()

TOY_CHUNK_SIZE = 250
TOY_CHUNK_OVERLAP = 20

toy_splitter = RecursiveCharacterTextSplitter(
    chunk_size=TOY_CHUNK_SIZE,
    chunk_overlap=TOY_CHUNK_OVERLAP,
    add_start_index=True,
)

toy_docs = toy_splitter.create_documents([toy_text])
for i, d in enumerate(toy_docs):
    d.metadata["chunk_index"] = i
    d.metadata["source"] = "toy"

toy_vs = FAISS.from_documents(toy_docs, emb)
toy_retriever = toy_vs.as_retriever(search_kwargs={"k": 1})

toy_query = "When did deep learning become prominent in AI?"

toy_baseline = [d.page_content for d in toy_retriever.invoke(toy_query)]
toy_enriched = retrieve_with_context_window(
    toy_docs,
    toy_retriever,
    toy_query,
    num_neighbors=1,
    chunk_overlap=TOY_CHUNK_OVERLAP,
)

print("Regular retrieval (toy):\n")
show_context(toy_baseline)

print("Retrieval with context window (toy):\n")
show_context(toy_enriched)

Regular retrieval (toy):

Context 1:
Deep Learning, a subset of machine learning using neural networks with many layers, began to show promising results in the early 2010s. The breakthrough came in 2012 when a deep neural network significantly outperformed other machine learning

Retrieval with context window (toy):

Context 1:
The late 1990s and 2000s marked the rise of machine learning approaches. Support Vector Machines (SVMs) and Random Forests became popular for various classification aDeep Learning, a subset of machine learning using neural networks with many layers, began to show promising results in the early 2010s. The breakthrough came in 2012 when a deep neural network significantly outperformed otmachine learning methods in the ImageNet competition.



## 7) 小结

- **Window-around-chunk 的输入**：一次检索命中的 chunk（通常 k 很小）
- **Window-around-chunk 的输出**：以命中 chunk 为中心的“固定窗口上下文”（前后各 N 个邻居）
- **它解决的问题**：让 LLM 看到更完整的段落/句子/表格片段，而不是单个碎片